# Análisis Exploratorio de Datos (EDA)
## Clasificación Automática de Estadios del Sueño — Sleep-EDF (ST7242J0)
**Maestría en Inteligencia Artificial — Grupo 7**

## 1. Importaciones y configuración

In [1]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import signal as sp_signal
from scipy.stats import kurtosis, skew
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

# Estilo global
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Paleta de colores por estadio (consistente en todo el notebook)
STAGE_COLORS = {
    'W':   '#e74c3c',   # Rojo
    'N1':  '#f39c12',   # Naranja
    'N2':  '#3498db',   # Azul
    'N3':  '#2c3e50',   # Azul oscuro
    'REM': '#27ae60',   # Verde
    'M':   '#95a5a6',   # Gris
}
STAGE_ORDER = ['W', 'N1', 'N2', 'N3', 'REM', 'M']
# Rutas del dataset versionado con DVC.
# Se resuelve la raiz del repositorio para que el notebook funcione
# sin importar el directorio de trabajo. Si data/ no existe: dvc pull
def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data.dvc").exists():
            return candidate
    raise FileNotFoundError(
        "No se encontro la raiz del repositorio: falta data.dvc en los directorios padre"
    )

BASE      = find_repo_root() / "data" / "sleep-telemetry"
PSG_PATH  = BASE / "ST7242J0-PSG.edf"
HYPO_PATH = BASE / "ST7242JO-Hypnogram.edf"

print('Librerías cargadas correctamente ✓')

Librerías cargadas correctamente ✓


## 2. Carga de datos

In [2]:
# ── Cargar señales PSG ──────────────────────────────────────────────────────
raw = mne.io.read_raw_edf(PSG_PATH, preload=True, verbose=False)
raw.pick(['EEG Fpz-Cz', 'EEG Pz-Oz'])   # Solo canales EEG

sfreq    = raw.info['sfreq']             # 100 Hz
n_samples = raw.get_data().shape[1]
duration_h = n_samples / sfreq / 3600

print(f'Canales cargados : {raw.ch_names}')
print(f'Frecuencia       : {sfreq} Hz')
print(f'Duración         : {duration_h:.2f} horas ({n_samples} muestras)')

# ── Cargar anotaciones del hipnograma ───────────────────────────────────────
ann = mne.read_annotations(HYPO_PATH)

# Mapeo R&K → AASM
LABEL_MAP = {
    'Sleep stage W': 'W',
    'Sleep stage 1': 'N1',
    'Sleep stage 2': 'N2',
    'Sleep stage 3': 'N3',
    'Sleep stage 4': 'N3',   # Fusión 3+4 → N3
    'Sleep stage R': 'REM',
    'Movement time': 'M',
}

# Expandir anotaciones (cada una puede cubrir múltiples épocas de 30 s)
epoch_records = []
for desc, onset, dur in zip(ann.description, ann.onset, ann.duration):
    label = LABEL_MAP.get(desc, '?')
    n_ep  = int(round(dur / 30))
    for i in range(n_ep):
        epoch_records.append({
            'epoch_idx': len(epoch_records),
            'onset_s'  : onset + i * 30,
            'label'    : label,
            'rk_label' : desc,
        })

df_epochs = pd.DataFrame(epoch_records)

# Separar épocas válidas (excluir M y ?)
df_valid  = df_epochs[~df_epochs['label'].isin(['M', '?'])].reset_index(drop=True)
df_excl   = df_epochs[ df_epochs['label'].isin(['M', '?'])].reset_index(drop=True)

print(f'\nTotal épocas anotadas : {len(df_epochs)}')
print(f'Épocas válidas (AASM) : {len(df_valid)}')
print(f'Épocas excluidas (M/?) : {len(df_excl)}')
print('\nDistribución de clases:')
print(df_valid['label'].value_counts().reindex([s for s in STAGE_ORDER if s != 'M']))

Canales cargados : ['EEG Fpz-Cz', 'EEG Pz-Oz']
Frecuencia       : 100.0 Hz
Duración         : 8.19 horas (2948000 muestras)

Total épocas anotadas : 940
Épocas válidas (AASM) : 939
Épocas excluidas (M/?) : 1

Distribución de clases:
label
W       17
N1      22
N2     591
N3     139
REM    170
Name: count, dtype: int64
